In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("C:/Users/Jabulani.Mcineka/workspace/Data-QC/Mental _health_dat_QC")
RAW_ACTIVITY_PATH = DATA_DIR / "22762_PSS.csv"
CURATED_PSS_PATH = DATA_DIR / "PSS Dataset.csv"

raw_df1 = pd.read_csv(RAW_ACTIVITY_PATH)
raw_df2 = pd.read_csv(CURATED_PSS_PATH)

df1_clean = raw_df1.rename(columns={
    "Participant ID": "participant_id_source",
    "Session Scheduled Time": "scheduled_time_source",
    "[1_SAQ] Question 1": "question1_response_source",
    "[2_SAQ] Question 2": "question2_response_source",
    "[3_SAQ] Question 3": "question3_response_source",
    "[4_SAQ] Question 4": "question4_response_source",
})
df2_clean = raw_df2.rename(columns={
    "IIntID": "iint_id_source",
    "Map_ID": "map_id_source",
    "sched_time": "scheduled_time_source",
    "saq1_22762": "question1_response_source",
    "saq2_22762": "question2_response_source",
    "saq3_22762": "question3_response_source",
    "saq4_22762": "question4_response_source",
})

assert len(df1_clean) == len(raw_df1)
assert len(df2_clean) == len(raw_df2)
{
    "big_dataset_rows": len(df1_clean),
    "mini_dataset_rows": len(df2_clean),
}

{'big_dataset_rows': 585, 'mini_dataset_rows': 554}

In [2]:
ANSWER_MAP = {
    "Never": 0,
    "Almost Never": 1,
    "Sometimes": 2,
    "Fairly Often": 3,
    "Very Often": 4,
    ".": pd.NA,
}

question_columns = ["question1", "question2", "question3", "question4"]
raw_answers_1 = df1_clean[[f"{name}_response_source" for name in question_columns]].apply(
    lambda column: column.str.split(" - ", n=1).str[-1]
)

df1_standard = pd.DataFrame({
    "big_row_id": raw_df1.index,
    "participant_id": df1_clean["participant_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(df1_clean["scheduled_time_source"], errors="coerce", utc=True).dt.date,
})
for name in question_columns:
    df1_standard[name] = raw_answers_1[f"{name}_response_source"].replace(".", pd.NA).map(ANSWER_MAP).astype("Int64")

df2_standard = pd.DataFrame({
    "mini_row_id": raw_df2.index,
    "iint_id": df2_clean["iint_id_source"].astype("string").str.strip(),
    "map_id_source": df2_clean["map_id_source"].astype("string").str.strip(),
    "scheduled_date": pd.to_datetime(df2_clean["scheduled_time_source"], format="%d-%b-%y", errors="coerce").dt.date,
})
for name in question_columns:
    df2_standard[name] = df2_clean[f"{name}_response_source"].replace(".", pd.NA).map(ANSWER_MAP).astype("Int64")

df2_standard["map_id_digits"] = df2_standard["map_id_source"].str.replace(r"[^0-9]", "", regex=True)
df2_standard["iint_id_prefix_match"] = [map_digits.startswith(iint_id) for map_digits, iint_id in zip(df2_standard["map_id_digits"], df2_standard["iint_id"])]
df2_standard["map_id_suffix"] = [map_digits[len(iint_id):] if match else "" for map_digits, iint_id, match in zip(df2_standard["map_id_digits"], df2_standard["iint_id"], df2_standard["iint_id_prefix_match"])]
df2_standard["participant_id_sequences"] = [tuple(suffix[position:position + 5] for position in range(0, len(suffix), 5)) if suffix and len(suffix) % 5 == 0 else () for suffix in df2_standard["map_id_suffix"]]
df2_standard["participant_id_count"] = df2_standard["participant_id_sequences"].str.len()
df2_standard["map_id_match"] = [prefix_match and bool(sequence) for prefix_match, sequence in zip(df2_standard["iint_id_prefix_match"], df2_standard["participant_id_sequences"]) ]
df2_standard["mapping_ambiguous"] = df2_standard["participant_id_count"] > 1

assert len(df1_standard) == len(raw_df1)
assert len(df2_standard) == len(raw_df2)
df1_standard.head(), df2_standard.head()

(   big_row_id participant_id scheduled_date  question1  question2  question3  \
 0           0         100921     2024-10-02       <NA>       <NA>       <NA>   
 1           1         100295     2024-10-02       <NA>       <NA>       <NA>   
 2           2          43760     2024-10-02          0          2          3   
 3           3          89593     2024-10-02          2          2          2   
 4           4          95065     2024-10-02          2          4          2   
 
    question4  
 0       <NA>  
 1       <NA>  
 2          4  
 3          3  
 4          3  ,
    mini_row_id iint_id map_id_source scheduled_date  question1  question2  \
 0            0    1294   129,459,466     2024-10-02       <NA>       <NA>   
 1            1    2221   222,143,781     2024-10-02          2          4   
 2            2    2221   222,143,781     2024-11-02          0          0   
 3            3    2221   222,143,781     2024-12-02          2          4   
 4            4    2221  

In [3]:
df2_for_match = df2_standard.copy()
df2_for_match["participant_id"] = df2_for_match["participant_id_sequences"].apply(list)
df2_for_match = df2_for_match.explode("participant_id", ignore_index=True)
df2_for_match["participant_id"] = df2_for_match["participant_id"].astype("string")
df1_standard["participant_id"] = df1_standard["participant_id"].astype("string")

comparison = df2_for_match.merge(df1_standard, on=["participant_id", "scheduled_date"], how="left", suffixes=("_mini", "_big"), indicator=True)
comparison["found_in_big"] = comparison["_merge"].eq("both")
for name in question_columns:
    comparison[f"{name}_match"] = comparison[f"{name}_mini"].eq(comparison[f"{name}_big"])
comparison["record_match"] = comparison["found_in_big"] & comparison[[f"{name}_match" for name in question_columns]].all(axis=1)
comparison.head()

,mini_row_id,iint_id,map_id_source,scheduled_date,question1_mini,question2_mini,question3_mini,question4_mini,map_id_digits,iint_id_prefix_match,...,question2_big,question3_big,question4_big,_merge,found_in_big,question1_match,question2_match,question3_match,question4_match,record_match
0,0,1294,"129,459,466",2024-10-02,<NA>,<NA>,<NA>,<NA>,129459466,True,...,<NA>,<NA>,<NA>,both,True,<NA>,<NA>,<NA>,<NA>,True
1,1,2221,"222,143,781",2024-10-02,2,4,3,2,222143781,True,...,4,3,2,both,True,True,True,True,True,True
2,2,2221,"222,143,781",2024-11-02,0,0,1,2,222143781,True,...,0,1,2,both,True,True,True,True,True,True
3,3,2221,"222,143,781",2024-12-02,2,4,4,3,222143781,True,...,4,4,3,both,True,True,True,True,True,True
4,4,2221,"222,143,781",2025-01-02,2,3,3,2,222143781,True,...,3,3,2,both,True,True,True,True,True,True


In [4]:
mini_row_status = comparison.groupby("mini_row_id", as_index=False)["found_in_big"].any().rename(columns={"found_in_big": "mini_row_found_in_big"})
mini_row_status["mini_row_status"] = mini_row_status["mini_row_found_in_big"].map({True: "found_in_big", False: "missing_from_big"})
comparison = comparison.merge(mini_row_status, on="mini_row_id", how="left")
mini_coverage = df2_standard.merge(mini_row_status, on="mini_row_id", how="left")
assert len(mini_coverage) == len(raw_df2)

comparison_summary = {
    "big_dataset_rows_preserved": len(df1_standard),
    "mini_dataset_rows_preserved": len(df2_standard),
    "mini_rows_found_in_big": int(mini_row_status["mini_row_found_in_big"].sum()),
    "mini_rows_missing_from_big": int((~mini_row_status["mini_row_found_in_big"]).sum()),
    "matched_participant_rows": int(comparison["found_in_big"].sum()),
    "fully_matching_participant_rows": int(comparison["record_match"].sum()),
    "multi_participant_map_ids": int((df2_standard["participant_id_count"] > 1).sum()),
}

mini_rows_found_in_big = mini_coverage.loc[mini_coverage["mini_row_found_in_big"]].reset_index(drop=True)
mini_rows_missing_from_big = mini_coverage.loc[~mini_coverage["mini_row_found_in_big"]].reset_index(drop=True)
fully_matching_rows = comparison.loc[comparison["record_match"]].reset_index(drop=True)
comparison_summary

{'big_dataset_rows_preserved': 585,
 'mini_dataset_rows_preserved': 554,
 'mini_rows_found_in_big': 548,
 'mini_rows_missing_from_big': 6,
 'matched_participant_rows': 548,
 'fully_matching_participant_rows': 548,
 'multi_participant_map_ids': 42}

In [5]:
print(f"Total mini PSS dataset rows: {comparison_summary['mini_dataset_rows_preserved']}")
display(df2_standard.head())
print(f"Total mini PSS rows found in big dataset: {comparison_summary['mini_rows_found_in_big']}")
display(mini_rows_found_in_big.head())
print(f"Total mini PSS rows missing from big dataset: {comparison_summary['mini_rows_missing_from_big']}")
display(mini_rows_missing_from_big.head())
print(f"Total fully matching PSS participant rows: {comparison_summary['fully_matching_participant_rows']}")
display(fully_matching_rows.head())

Total mini PSS dataset rows: 554


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous
0,0,1294,"129,459,466",2024-10-02,<NA>,<NA>,<NA>,<NA>,129459466,True,59466,"(59466,)",1,True,False
1,1,2221,"222,143,781",2024-10-02,2,4,3,2,222143781,True,43781,"(43781,)",1,True,False
2,2,2221,"222,143,781",2024-11-02,0,0,1,2,222143781,True,43781,"(43781,)",1,True,False
3,3,2221,"222,143,781",2024-12-02,2,4,4,3,222143781,True,43781,"(43781,)",1,True,False
4,4,2221,"222,143,781",2025-01-02,2,3,3,2,222143781,True,43781,"(43781,)",1,True,False


Total mini PSS rows found in big dataset: 548


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,0,1294,"129,459,466",2024-10-02,<NA>,<NA>,<NA>,<NA>,129459466,True,59466,"(59466,)",1,True,False,True,found_in_big
1,1,2221,"222,143,781",2024-10-02,2,4,3,2,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
2,2,2221,"222,143,781",2024-11-02,0,0,1,2,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
3,3,2221,"222,143,781",2024-12-02,2,4,4,3,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big
4,4,2221,"222,143,781",2025-01-02,2,3,3,2,222143781,True,43781,"(43781,)",1,True,False,True,found_in_big


Total mini PSS rows missing from big dataset: 6


,mini_row_id,iint_id,map_id_source,scheduled_date,question1,question2,question3,question4,map_id_digits,iint_id_prefix_match,map_id_suffix,participant_id_sequences,participant_id_count,map_id_match,mapping_ambiguous,mini_row_found_in_big,mini_row_status
0,454,130789,"1,307,895,895,089,590",2024-10-02,2,2,2,3,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
1,455,130789,"1,307,895,895,089,590",2024-11-02,2,2,2,2,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
2,456,130789,"1,307,895,895,089,590",2024-12-02,2,2,3,3,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
3,457,130789,"1,307,895,895,089,590",2025-01-02,1,2,2,2,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big
4,458,130789,"1,307,895,895,089,590",2025-02-02,2,2,2,2,1307895895089590,True,5895089590,"(58950, 89590)",2,True,True,False,missing_from_big


Total fully matching PSS participant rows: 548


,mini_row_id,iint_id,map_id_source,scheduled_date,question1_mini,question2_mini,question3_mini,question4_mini,map_id_digits,iint_id_prefix_match,...,question4_big,_merge,found_in_big,question1_match,question2_match,question3_match,question4_match,record_match,mini_row_found_in_big,mini_row_status
0,0,1294,"129,459,466",2024-10-02,<NA>,<NA>,<NA>,<NA>,129459466,True,...,<NA>,both,True,<NA>,<NA>,<NA>,<NA>,True,True,found_in_big
1,1,2221,"222,143,781",2024-10-02,2,4,3,2,222143781,True,...,2,both,True,True,True,True,True,True,True,found_in_big
2,2,2221,"222,143,781",2024-11-02,0,0,1,2,222143781,True,...,2,both,True,True,True,True,True,True,True,found_in_big
3,3,2221,"222,143,781",2024-12-02,2,4,4,3,222143781,True,...,3,both,True,True,True,True,True,True,True,found_in_big
4,4,2221,"222,143,781",2025-01-02,2,3,3,2,222143781,True,...,2,both,True,True,True,True,True,True,True,found_in_big
